# 🌪️ Disaster Tweets — EDA, Classical Baselines & RoBERTa Ensemble

**Competition:** [Natural Language Processing with Disaster Tweets](https://www.kaggle.com/competitions/nlp-getting-started)
**Task:** binary classification — is a tweet about a real disaster (`1`) or not (`0`)?
**Metric:** F1-score on the positive class.

This notebook is organised as a small, modular pipeline. Every model plugs into the same cross-validation
helper, so results are directly comparable and can be blended at the end.

| # | Section | What happens |
|---|---------|--------------|
| 1 | Setup | config, seeds, paths |
| 2 | EDA | class balance, missing values, length, keywords, label noise |
| 3 | Preprocessing | text cleaning, label-noise fix, meta features, folds |
| 4 | Baseline A | TF-IDF (word + char) → Logistic Regression |
| 5 | Baseline B | TF-IDF + meta features → LightGBM |
| 6 | Language model | fine-tuned `roberta-base` (5-fold) |
| 7 | Ensemble | weighted blend + threshold tuned on out-of-fold predictions |
| 8 | Submission | `submission.csv` |

> **Run settings on Kaggle:** Accelerator = *GPU T4 x2* (or P100) and *Internet = On* (needed to download `roberta-base`).
> Without a GPU/internet the notebook still runs end-to-end — the transformer section is skipped automatically.

## 1. Setup

In [ ]:
import gc
import html
import os
import random
import re
import warnings
from itertools import product
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline, make_union

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_colwidth", 120)


class CFG:
    seed = 42
    n_folds = 5
    kaggle_dir = Path("/kaggle/input/nlp-getting-started")
    data_dir = kaggle_dir if kaggle_dir.exists() else Path("data")
    out_dir = Path(".")

    # transformer
    run_transformer = True          # auto-disabled below if no GPU / transformers
    model_name = "roberta-base"
    max_len = 80
    batch_size = 32
    epochs = 3
    lr = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.1


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


seed_everything(CFG.seed)
print("data dir:", CFG.data_dir.resolve())

## 2. Exploratory Data Analysis

In [ ]:
train = pd.read_csv(CFG.data_dir / "train.csv")
test = pd.read_csv(CFG.data_dir / "test.csv")
print(f"train: {train.shape}  |  test: {test.shape}")
train.head()

### 2.1 Target balance and missing values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train["target"].value_counts().sort_index().plot.bar(ax=axes[0], color=["#4C72B0", "#C44E52"])
axes[0].set_title("Target distribution")
axes[0].set_xticklabels(["0 – not disaster", "1 – disaster"], rotation=0)

missing = pd.DataFrame({
    "train": train[["keyword", "location"]].isna().mean(),
    "test": test[["keyword", "location"]].isna().mean(),
}) * 100
missing.plot.bar(ax=axes[1], rot=0)
axes[1].set_title("Missing values (%)")
plt.tight_layout()
plt.show()

print(train["target"].value_counts(normalize=True).round(3))

The classes are mildly imbalanced (~57 / 43), so F1 is sensitive to the decision threshold — we will tune it on
out-of-fold predictions instead of blindly using 0.5. `location` is ~33% missing and free-text noise; `keyword`
is almost always present and turns out to be very informative.

### 2.2 Text length and tweet "style"

In [ ]:
def count_pattern(s: pd.Series, pattern: str) -> pd.Series:
    return s.str.count(pattern)


eda = train.assign(
    char_len=train["text"].str.len(),
    word_count=train["text"].str.split().str.len(),
    n_urls=count_pattern(train["text"], r"https?://"),
    n_hashtags=count_pattern(train["text"], r"#\w+"),
    n_mentions=count_pattern(train["text"], r"@\w+"),
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["char_len", "word_count", "n_urls"]):
    sns.histplot(data=eda, x=col, hue="target", stat="density", common_norm=False,
                 discrete=(col == "n_urls"), element="step", ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

eda.groupby("target")[["char_len", "word_count", "n_urls", "n_hashtags", "n_mentions"]].mean().round(2)

Disaster tweets tend to be slightly longer and contain more links (news-like), while non-disaster tweets have more
`@mentions` (conversational). These stylistic signals motivate the *meta features* used by the LightGBM model.

### 2.3 Keywords

In [ ]:
kw = (train.dropna(subset=["keyword"])
      .assign(keyword=lambda d: d["keyword"].str.replace("%20", " "))
      .groupby("keyword")["target"].agg(["mean", "count"])
      .query("count >= 20")
      .sort_values("mean"))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
kw.head(15)["mean"].plot.barh(ax=axes[0], color="#4C72B0")
axes[0].set_title("Keywords least associated with disasters")
kw.tail(15)["mean"].plot.barh(ax=axes[1], color="#C44E52")
axes[1].set_title("Keywords most associated with disasters")
for ax in axes:
    ax.set_xlabel("disaster rate")
    ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

print(f"{train['keyword'].nunique()} unique keywords | "
      f"{(train['keyword'].dropna().isin(test['keyword'].dropna())).mean():.1%} of train keywords also appear in test")

Keywords like *wreckage*, *derailment* or *debris* are almost always real disasters, while *body bags*, *ruin* or
*screaming* are mostly figurative. The same keyword vocabulary appears in train and test, so we prepend the keyword
to the text for every model.

### 2.4 Label noise: duplicated tweets with conflicting targets

In [ ]:
dup = train.groupby("text")["target"].agg(["nunique", "count"])
conflicting = dup.query("nunique > 1")
print(f"duplicated texts: {(dup['count'] > 1).sum()} | with conflicting labels: {len(conflicting)}")
train[train["text"].isin(conflicting.index)].sort_values("text")[["text", "target"]].head(10)

The same tweet appears with both labels — pure annotation noise. In preprocessing we resolve these by majority vote.

## 3. Preprocessing

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """Light normalisation that keeps useful signal (we replace URLs/mentions with tokens instead of deleting them)."""
    text = html.unescape(text)               # &amp; -> &
    text = URL_RE.sub(" http ", text)
    text = MENTION_RE.sub(" @user ", text)
    text = text.replace("#", " ")            # #earthquake -> earthquake
    text = text.encode("ascii", "ignore").decode()  # drop mojibake like Ûª
    return SPACE_RE.sub(" ", text).strip()


def build_input(df: pd.DataFrame) -> pd.Series:
    keyword = df["keyword"].fillna("").str.replace("%20", " ")
    return (keyword + " | " + df["text"].map(clean_text)).str.strip(" |")


def fix_conflicting_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    mean_target = df.groupby("text")["target"].transform("mean")
    noisy = (mean_target > 0) & (mean_target < 1)
    df.loc[noisy, "target"] = (mean_target[noisy] > 0.5).astype(int)
    print(f"relabelled {noisy.sum()} rows")
    return df


def add_meta_features(df: pd.DataFrame) -> pd.DataFrame:
    t = df["text"]
    words = t.str.split()
    return pd.DataFrame({
        "char_len": t.str.len(),
        "word_count": words.str.len(),
        "mean_word_len": words.map(lambda w: np.mean([len(x) for x in w])),
        "n_urls": t.str.count(r"https?://"),
        "n_hashtags": t.str.count(r"#\w+"),
        "n_mentions": t.str.count(r"@\w+"),
        "n_exclaim": t.str.count("!"),
        "n_question": t.str.count(r"\?"),
        "upper_ratio": t.map(lambda s: sum(c.isupper() for c in s) / max(len(s), 1)),
        "digit_ratio": t.map(lambda s: sum(c.isdigit() for c in s) / max(len(s), 1)),
        "has_location": df["location"].notna().astype(int),
    }, index=df.index)


train = fix_conflicting_labels(train)
for df in (train, test):
    df["clean"] = build_input(df)

META_COLS = list(add_meta_features(train.head()).columns)
train[META_COLS] = add_meta_features(train)
test[META_COLS] = add_meta_features(test)

# shared stratified folds -> every model sees exactly the same splits (needed for a fair blend)
train["fold"] = -1
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
for fold, (_, va_idx) in enumerate(skf.split(train, train["target"])):
    train.loc[va_idx, "fold"] = fold

train[["text", "clean", "target", "fold"]].head()

### 3.1 Cross-validation helpers

Each model is just a function `fit_predict(train_df, valid_df, test_df) -> (valid_probs, test_probs)`.
`run_cv` handles the fold loop, out-of-fold (OOF) bookkeeping and test-set averaging.

In [ ]:
y = train["target"].values
RESULTS: dict[str, dict] = {}
OOF: dict[str, np.ndarray] = {}
TEST: dict[str, np.ndarray] = {}


def best_threshold(y_true: np.ndarray, probs: np.ndarray) -> tuple[float, float]:
    thresholds = np.arange(0.20, 0.81, 0.01)
    scores = [f1_score(y_true, probs >= t) for t in thresholds]
    i = int(np.argmax(scores))
    return float(thresholds[i]), float(scores[i])


def run_cv(name: str, fit_predict) -> None:
    oof = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    for fold in range(CFG.n_folds):
        tr = train[train["fold"] != fold]
        va = train[train["fold"] == fold]
        p_va, p_te = fit_predict(tr, va, test)
        oof[va.index] = p_va
        test_pred += p_te / CFG.n_folds
        print(f"[{name}] fold {fold}: F1@0.5 = {f1_score(va['target'], p_va >= 0.5):.4f}")

    thr, f1_tuned = best_threshold(y, oof)
    RESULTS[name] = {"oof_f1@0.5": f1_score(y, oof >= 0.5), "best_thr": thr, "oof_f1@best": f1_tuned}
    OOF[name], TEST[name] = oof, test_pred
    np.save(CFG.out_dir / f"oof_{name}.npy", oof)
    np.save(CFG.out_dir / f"test_{name}.npy", test_pred)
    print(f"[{name}] OOF F1@0.5 = {RESULTS[name]['oof_f1@0.5']:.4f} | "
          f"OOF F1@{thr:.2f} = {f1_tuned:.4f}\n")


def save_submission(probs: np.ndarray, threshold: float, filename: str) -> pd.DataFrame:
    sub = pd.DataFrame({"id": test["id"], "target": (probs >= threshold).astype(int)})
    sub.to_csv(CFG.out_dir / filename, index=False)
    return sub

## 4. Baseline A — TF-IDF (word + char n-grams) → Logistic Regression

Fast, strong and interpretable. Character n-grams make it robust to typos, hashtags glued to words and slang.

In [ ]:
def tfidf_logreg(tr: pd.DataFrame, va: pd.DataFrame, te: pd.DataFrame):
    model = make_pipeline(
        make_union(
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode"),
            TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
        ),
        LogisticRegression(C=2.0, max_iter=3000, solver="liblinear"),
    )
    model.fit(tr["clean"], tr["target"])
    return model.predict_proba(va["clean"])[:, 1], model.predict_proba(te["clean"])[:, 1]


run_cv("tfidf_logreg", tfidf_logreg)

## 5. Baseline B — LightGBM on TF-IDF + meta features

Gradient boosting can combine sparse n-gram evidence with the dense stylistic features found in the EDA.
A fixed number of trees is used (no early stopping on the validation fold) to keep the OOF estimate honest.

In [ ]:
LGB_PARAMS = dict(
    n_estimators=700,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=10,
    colsample_bytree=0.3,
    subsample=0.8,
    subsample_freq=1,
    reg_lambda=1.0,
    random_state=CFG.seed,
    verbose=-1,
)


def tfidf_lgbm(tr: pd.DataFrame, va: pd.DataFrame, te: pd.DataFrame):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20_000, sublinear_tf=True)
    to_matrix = lambda df, fit=False: sparse.hstack([
        vec.fit_transform(df["clean"]) if fit else vec.transform(df["clean"]),
        sparse.csr_matrix(df[META_COLS].values.astype(np.float32)),
    ]).tocsr().astype(np.float32)

    X_tr = to_matrix(tr, fit=True)
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_tr, tr["target"])
    return model.predict_proba(to_matrix(va))[:, 1], model.predict_proba(to_matrix(te))[:, 1]


run_cv("tfidf_lgbm", tfidf_lgbm)

## 6. Language model — fine-tuned RoBERTa

`roberta-base` was pre-trained on a large web corpus (including informal text), so it understands figurative usage
("my phone just *exploded* with notifications") far better than bag-of-words models.
A plain PyTorch loop is used instead of `Trainer` to keep the code transparent and version-proof.

In [ ]:
try:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              get_linear_schedule_with_warmup)
    from transformers.utils import logging as hf_logging

    hf_logging.set_verbosity_error()
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = None

if DEVICE != "cuda":
    print(f"No GPU / transformers available (device={DEVICE}) -> skipping the transformer section.")
    CFG.run_transformer = False

if CFG.run_transformer:
    try:
        tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
    except OSError as e:
        print("Could not download the model (is Internet enabled?) -> skipping.\n", e)
        CFG.run_transformer = False

In [ ]:
def make_loader(texts: pd.Series, labels=None, shuffle: bool = False) -> "DataLoader":
    enc = tokenizer(list(texts), truncation=True, padding="max_length",
                    max_length=CFG.max_len, return_tensors="pt")
    tensors = [enc["input_ids"], enc["attention_mask"]]
    if labels is not None:
        tensors.append(torch.tensor(np.asarray(labels), dtype=torch.long))
    return DataLoader(TensorDataset(*tensors), batch_size=CFG.batch_size, shuffle=shuffle)


def predict_proba(model, loader) -> np.ndarray:
    model.eval()
    probs = []
    with torch.no_grad():
        for batch in loader:
            ids, mask = batch[0].to(DEVICE), batch[1].to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(input_ids=ids, attention_mask=mask).logits
            probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs)


def roberta(tr: pd.DataFrame, va: pd.DataFrame, te: pd.DataFrame):
    model = AutoModelForSequenceClassification.from_pretrained(CFG.model_name, num_labels=2).to(DEVICE)
    tr_loader = make_loader(tr["clean"], tr["target"], shuffle=True)
    va_loader = make_loader(va["clean"])
    te_loader = make_loader(te["clean"])

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    total_steps = len(tr_loader) * CFG.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG.warmup_ratio * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(CFG.epochs):
        model.train()
        for ids, mask, labels in tr_loader:
            ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        p_va = predict_proba(model, va_loader)
        print(f"   epoch {epoch + 1}: valid F1@0.5 = {f1_score(va['target'], p_va >= 0.5):.4f}")

    # fixed number of epochs (no epoch picking on the validation fold) -> honest OOF
    p_te = predict_proba(model, te_loader)
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return p_va, p_te


if CFG.run_transformer:
    seed_everything(CFG.seed)
    run_cv("roberta", roberta)

## 7. Ensemble

Grid search over blend weights (step 0.05, summing to 1) and the decision threshold, maximising **OOF F1**.
Because all models share the same folds, the OOF predictions are directly comparable.

In [ ]:
def blend_search(names: list[str], step: float = 0.05):
    grid = np.round(np.arange(0, 1 + 1e-9, step), 2)
    best = {"f1": -1.0}
    for w in product(grid, repeat=len(names)):
        if not np.isclose(sum(w), 1.0):
            continue
        probs = sum(wi * OOF[n] for wi, n in zip(w, names))
        thr, f1 = best_threshold(y, probs)
        if f1 > best["f1"]:
            best = {"weights": {n: float(wi) for n, wi in zip(names, w)}, "thr": thr, "f1": f1}
    return best


names = list(OOF)
best = blend_search(names)
print("best weights:", best["weights"])
print(f"threshold = {best['thr']:.2f} | OOF F1 = {best['f1']:.4f}")

blend_oof = sum(w * OOF[n] for n, w in best["weights"].items())
blend_test = sum(w * TEST[n] for n, w in best["weights"].items())
RESULTS["ensemble"] = {"oof_f1@0.5": f1_score(y, blend_oof >= 0.5), "best_thr": best["thr"], "oof_f1@best": best["f1"]}

In [ ]:
corr = pd.DataFrame(OOF).corr()
plt.figure(figsize=(1.6 * len(names) + 2, 1.4 * len(names) + 1))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="Blues", vmin=0.5, vmax=1)
plt.title("Correlation between OOF predictions")
plt.show()

Lower correlation between models means more complementary errors — that is where blending helps the most.

## 8. Results & submission

In [ ]:
results = pd.DataFrame(RESULTS).T.sort_values("oof_f1@best", ascending=False)
results.round(4)

In [ ]:
# one file per model (handy to compare public LB scores) + the final blend as submission.csv
for name in OOF:
    save_submission(TEST[name], RESULTS[name]["best_thr"], f"submission_{name}.csv")

submission = save_submission(blend_test, best["thr"], "submission.csv")
print(submission["target"].value_counts(normalize=True).round(3))
submission.head()

## 9. Takeaways

* **Threshold tuning matters for F1** — moving away from 0.5 gives a free boost for every model.
* **Keyword + light cleaning** beats aggressive cleaning: URLs and mentions carry signal, so they are replaced by
  tokens rather than deleted.
* **Label noise** exists (duplicated tweets with opposite labels); fixing it by majority vote cleans the training signal.
* **Pre-trained language models** capture figurative language that n-gram models miss; blending them with the
  classical baselines adds diversity.

**Next ideas:** `deberta-v3-base`/`bertweet-base`, multi-seed averaging, pseudo-labelling the test set,
and error analysis on the most confident OOF mistakes.

> ⚠️ The public leaderboard of this competition contains many perfect (1.000) scores from people who submitted
> the publicly available labels. Realistic honest scores are roughly **0.80–0.85**.